# 02 -- Factor Engineering
Produce `factors_monthly.parquet` with standardized cross-sectional factors.

In [1]:
import warnings, os
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

art = pd.read_parquet('../data/processed/panel_artemis_daily.parquet')
uni = pd.read_parquet('../data/processed/universe_monthly.parquet')

art['date'] = pd.to_datetime(art['date'])
uni['date'] = pd.to_datetime(uni['date'])
art = art.sort_values(['symbol', 'date'])

rebalance_dates = sorted(uni['date'].unique())
print(f"Artemis: {art.shape} | Universe: {uni.shape}")
print(f"Rebalance dates: {len(rebalance_dates)}  ({rebalance_dates[0].date()} -> {rebalance_dates[-1].date()})")

def make_wide(df, col):
    return df.pivot_table(index='date', columns='symbol', values=col, aggfunc='last')

price_w = make_wide(art, 'price')
fees_w  = make_wide(art, 'fees')
rev_w   = make_wide(art, 'revenue')
tvl_w   = make_wide(art, 'tvl')
dau_w   = make_wide(art, 'dau')
txns_w  = make_wide(art, 'txns')
mc_w    = make_wide(art, 'mc')
print("Wide tables built.")


Artemis: (160487, 9) | Universe: (2400, 4)
Rebalance dates: 48  (2021-01-31 -> 2024-12-31)


Wide tables built.


In [2]:
## Helper functions (no lookahead: all slices end at or before rebalance date t)

def last_val(wide, t, cols):
    # Last non-null value on or before date t for each column.
    sub = wide.reindex(columns=cols).loc[:t]
    if len(sub) == 0:
        return pd.Series(np.nan, index=cols)
    return sub.ffill().iloc[-1]


def window_mean(wide, t_end, n_days, cols, min_obs=15):
    # Mean over n_days ending at t_end; NaN if fewer than min_obs non-null.
    t_start = t_end - pd.Timedelta(days=n_days - 1)
    sub = wide.reindex(columns=cols).loc[t_start:t_end]
    cnt = sub.notna().sum()
    m = sub.mean()
    m[cnt < min_obs] = np.nan
    return m


def window_std(wide, t_end, n_days, cols, min_obs=15):
    t_start = t_end - pd.Timedelta(days=n_days - 1)
    sub = wide.reindex(columns=cols).loc[t_start:t_end]
    cnt = sub.notna().sum()
    s = sub.std()
    s[cnt < min_obs] = np.nan
    return s


def realized_vol(pw, t_end, n_days, cols, min_obs=10):
    # Annualised realised vol from log daily returns.
    t_start = t_end - pd.Timedelta(days=n_days - 1)
    sub = pw.reindex(columns=cols).loc[t_start:t_end]
    log_ret = np.log(sub).diff()
    cnt = log_ret.notna().sum()
    s = log_ret.std() * np.sqrt(252)
    s[cnt < min_obs] = np.nan
    return s


def safe_ratio(num, denom):
    d = denom.copy().astype(float)
    d[d == 0] = np.nan
    return num / d


In [3]:
## Factor computation loop -- no future data used (all windows end at t)
FACTOR_COLS = [
    'mom_1m', 'mom_3m', 'mom_6m', 'vol_30d',
    'fees_mc', 'rev_mc', 'fees_growth_30d', 'rev_growth_30d',
    'dau_growth_30d', 'txns_growth_30d', 'dau_zscore',
    'tvl_mc', 'tvl_growth_30d',
]

all_records = []

for t in rebalance_dates:
    uni_syms = uni[uni['date'] == t]['symbol'].tolist()
    uni_syms = [s for s in uni_syms if s in price_w.columns]

    # Group A: Momentum
    p_now = last_val(price_w, t, uni_syms)
    p_30  = last_val(price_w, t - pd.Timedelta(30),  uni_syms)
    p_90  = last_val(price_w, t - pd.Timedelta(90),  uni_syms)
    p_180 = last_val(price_w, t - pd.Timedelta(180), uni_syms)

    mom_1m = safe_ratio(p_now, p_30)  - 1
    mom_3m = safe_ratio(p_now, p_90)  - 1
    mom_6m = safe_ratio(p_now, p_180) - 1
    vol_30 = realized_vol(price_w, t, 31, uni_syms, min_obs=10)

    # Group B: Fundamentals
    mc_now = last_val(mc_w, t, uni_syms)

    f_30    = window_mean(fees_w, t,                        30, uni_syms)
    f_lag30 = window_mean(fees_w, t - pd.Timedelta(30),    30, uni_syms)
    fees_mc        = safe_ratio(f_30, mc_now)
    fees_growth_30 = safe_ratio(f_30, f_lag30) - 1

    r_30    = window_mean(rev_w, t,                      30, uni_syms)
    r_lag30 = window_mean(rev_w, t - pd.Timedelta(30),  30, uni_syms)
    rev_mc        = safe_ratio(r_30, mc_now)
    rev_growth_30 = safe_ratio(r_30, r_lag30) - 1

    # Group C: Usage
    d_30    = window_mean(dau_w, t,                      30, uni_syms)
    d_lag30 = window_mean(dau_w, t - pd.Timedelta(30),  30, uni_syms)
    dau_growth_30 = safe_ratio(d_30, d_lag30) - 1

    d_90_mean = window_mean(dau_w, t, 90, uni_syms, min_obs=20)
    d_90_std  = window_std(dau_w,  t, 90, uni_syms, min_obs=20)
    dau_zscore = safe_ratio(d_30 - d_90_mean, d_90_std)

    x_30    = window_mean(txns_w, t,                     30, uni_syms)
    x_lag30 = window_mean(txns_w, t - pd.Timedelta(30), 30, uni_syms)
    txns_growth_30 = safe_ratio(x_30, x_lag30) - 1

    # Group D: TVL
    tv_30    = window_mean(tvl_w, t,                     30, uni_syms)
    tv_lag30 = window_mean(tvl_w, t - pd.Timedelta(30), 30, uni_syms)
    tvl_mc        = safe_ratio(tv_30, mc_now)
    tvl_growth_30 = safe_ratio(tv_30, tv_lag30) - 1

    df_t = pd.DataFrame({
        'symbol': uni_syms, 'date': t,
        'mom_1m':          mom_1m.reindex(uni_syms).values,
        'mom_3m':          mom_3m.reindex(uni_syms).values,
        'mom_6m':          mom_6m.reindex(uni_syms).values,
        'vol_30d':         vol_30.reindex(uni_syms).values,
        'fees_mc':         fees_mc.reindex(uni_syms).values,
        'rev_mc':          rev_mc.reindex(uni_syms).values,
        'fees_growth_30d': fees_growth_30.reindex(uni_syms).values,
        'rev_growth_30d':  rev_growth_30.reindex(uni_syms).values,
        'dau_growth_30d':  dau_growth_30.reindex(uni_syms).values,
        'txns_growth_30d': txns_growth_30.reindex(uni_syms).values,
        'dau_zscore':      dau_zscore.reindex(uni_syms).values,
        'tvl_mc':          tvl_mc.reindex(uni_syms).values,
        'tvl_growth_30d':  tvl_growth_30.reindex(uni_syms).values,
    })
    all_records.append(df_t)

factors_raw = pd.concat(all_records, ignore_index=True)
print(f"Raw factors: {factors_raw.shape}")
print("Null rates (raw):")
print(factors_raw[FACTOR_COLS].isnull().mean().round(3).to_string())


Raw factors: (2400, 15)
Null rates (raw):
mom_1m             0.000
mom_3m             0.000
mom_6m             0.000
vol_30d            0.001
fees_mc            0.524
rev_mc             0.447
fees_growth_30d    0.536
rev_growth_30d     0.549
dau_growth_30d     0.427
txns_growth_30d    0.547
dau_zscore         0.428
tvl_mc             0.909
tvl_growth_30d     0.924


In [4]:
## Standardization: winsorize at 1/99% -> z-score -> fill NaN = 0

factors_std = factors_raw.copy()

for col in FACTOR_COLS:
    for t, idx in factors_raw.groupby('date').groups.items():
        s = factors_raw.loc[idx, col].astype(float)
        if s.notna().sum() < 3:
            factors_std.loc[idx, col] = 0.0
            continue
        q01, q99 = s.quantile(0.01), s.quantile(0.99)
        s = s.clip(q01, q99)
        mu, sigma = s.mean(), s.std()
        if sigma > 1e-10:
            s = (s - mu) / sigma
        else:
            s = pd.Series(0.0, index=s.index)
        factors_std.loc[idx, col] = s.fillna(0.0).values

print("Standardized factors -- mean / std check (should be ~0 / 1 per date):")
for col in FACTOR_COLS:
    mu  = factors_std.groupby('date')[col].mean().mean()
    sig = factors_std.groupby('date')[col].std().mean()
    print(f"  {col:20s}  mean={mu:+.3f}  std={sig:.3f}")


Standardized factors -- mean / std check (should be ~0 / 1 per date):
  mom_1m                mean=+0.000  std=1.000
  mom_3m                mean=+0.000  std=1.000
  mom_6m                mean=+0.000  std=1.000
  vol_30d               mean=-0.000  std=1.000
  fees_mc               mean=-0.000  std=0.680
  rev_mc                mean=+0.000  std=0.736
  fees_growth_30d       mean=+0.000  std=0.670
  rev_growth_30d        mean=-0.000  std=0.659
  dau_growth_30d        mean=-0.000  std=0.749
  txns_growth_30d       mean=+0.000  std=0.661
  dau_zscore            mean=+0.000  std=0.749
  tvl_mc                mean=-0.000  std=0.264
  tvl_growth_30d        mean=-0.000  std=0.235


In [5]:
out_path = '../data/processed/factors_monthly.parquet'
factors_std.to_parquet(out_path, index=False)
print(f"Saved: {out_path}  shape={factors_std.shape}")
print("\nSample (first rebalance date):")
first_t = factors_std['date'].min()
print(factors_std[factors_std['date'] == first_t][['symbol'] + FACTOR_COLS[:4]].head())
print("\n>> Factor engineering complete.")


Saved: ../data/processed/factors_monthly.parquet  shape=(2400, 15)

Sample (first rebalance date):
  symbol    mom_1m    mom_3m    mom_6m   vol_30d
0    BTC -0.548859 -0.548859 -0.548859 -0.950188
1    ETH -0.643515 -0.643515 -0.643515 -0.187345
2    XRP  1.751344  1.751344  1.751344  0.444931
3    DOT -0.346494 -0.346494 -0.346494  0.805173
4    ADA -0.653081 -0.653081 -0.653081  0.126129

>> Factor engineering complete.
